# Analysing the processed data

A working surface for investigating `data/processed/`. Distinct from `explore_data.ipynb`, which checks that the *pipeline* is behaving; this one asks questions of the *data*.

It builds a **semantic view** over the Parquet files, attaching the datamap labels to the codes. `indicator` (*what* is measured, e.g. `E_AI_TANY`) and `unit` (*of what population*, e.g. `PC_ENT`) are separate columns in the Parquet, so either can be filtered on its own. Two further fields come from the datamap's `unit` block and guard the two easiest ways to be confidently wrong here:

| Field | Why it matters |
|---|---|
| `base` | Every value is a percentage — but of **18 different base populations**. Two rows can both read `45.7` and mean different things. Only compare values sharing a base. |
| `weightable` | `enterprise_count` counts *all* enterprises, so only `PC_ENT` may be multiplied by it. Weighting anything else invents a number. |

Scope: 34 countries, 14 firm datasets, 2015–2025. 291 indicator codes x 18 units = 810 distinct metrics.

In [1]:
import json
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

PROCESSED = Path("data/processed")

firm = pd.read_parquet(PROCESSED / "firm_level.parquet")
individual = pd.read_parquet(PROCESSED / "individual_level.parquet")
firm_dm = json.loads((PROCESSED / "firm_level.datamap.json").read_text(encoding="utf-8"))
ind_dm = json.loads((PROCESSED / "individual_level.datamap.json").read_text(encoding="utf-8"))

print(f"firm_level       {len(firm):>10,} rows | {firm.geo.nunique()} countries | {firm.dataset.nunique()} datasets")
print(f"individual_level {len(individual):>10,} rows | {individual.geo.nunique()} countries | {individual.dataset.nunique()} datasets")

firm_level        1,351,527 rows | 34 countries | 14 datasets
individual_level    561,666 rows | 34 countries | 2 datasets


In [2]:
con = duckdb.connect()


def _codes_frame(dm, column, value_name):
    """Turn a datamap {code: label} block into a joinable frame."""
    codes = dm["columns"].get(column, {}).get("codes", {})
    return pd.DataFrame(list(codes.items()), columns=[column, value_name])


def _unit_frame(dm):
    """The unit block, whose entries carry the base population and the weighting guard."""
    rows = [
        (code, m.get("label"), m.get("base"), bool(m.get("weightable", False)))
        for code, m in dm["columns"]["unit"]["codes"].items()
    ]
    return pd.DataFrame(rows, columns=["unit", "unit_label", "base", "weightable"])


con.register("firm_raw", firm)
con.register("ind_raw", individual)
con.register("f_ind", _codes_frame(firm_dm, "indicator", "indic_label"))
con.register("f_unit", _unit_frame(firm_dm))
con.register("f_geo", _codes_frame(firm_dm, "geo", "geo_label"))
con.register("f_size", _codes_frame(firm_dm, "size_emp", "size_label"))
con.register("f_nace", _codes_frame(firm_dm, "nace_r2", "nace_label"))
con.register("i_ind", _codes_frame(ind_dm, "indicator", "indic_label"))
con.register("i_unit", _unit_frame(ind_dm))
con.register("i_type", _codes_frame(ind_dm, "ind_type", "ind_type_label"))

con.execute("""
CREATE OR REPLACE VIEW firm AS
SELECT r.indicator, m.indic_label,
       r.unit, u.unit_label, u.base, u.weightable,
       r.geo, g.geo_label, r.time,
       r.size_emp, s.size_label,
       r.nace_r2, n.nace_label,
       r.value, r.enterprise_count, r.dataset
FROM firm_raw r
LEFT JOIN f_ind  m USING (indicator)
LEFT JOIN f_unit u USING (unit)
LEFT JOIN f_geo  g USING (geo)
LEFT JOIN f_size s USING (size_emp)
LEFT JOIN f_nace n USING (nace_r2)
""")

con.execute("""
CREATE OR REPLACE VIEW people AS
SELECT r.indicator, m.indic_label,
       r.unit, u.unit_label, u.base,
       r.geo, r.time,
       r.ind_type, t.ind_type_label, r.sex, r.age, r.education,
       r.value, r.dataset
FROM ind_raw r
LEFT JOIN i_ind  m USING (indicator)
LEFT JOIN i_unit u USING (unit)
LEFT JOIN i_type t USING (ind_type)
""")


def q(sql, warn=True):
    """Run SQL and, unless told otherwise, warn when a result mixes base populations."""
    df = con.execute(sql).df()
    if warn and "base" in df.columns:
        bases = df["base"].dropna().unique()
        if len(bases) > 1:
            print(f"!! MIXED BASES ({len(bases)}): {list(bases)}")
            print("   These values are percentages of different populations - not comparable.")
    return df


print("views ready:  firm  |  people")

views ready:  firm  |  people


## The base-population problem

Everything here is a percentage, so nothing looks wrong when two incompatible metrics are plotted together. This is what the `base` column exists to prevent.

In [3]:
q("""
SELECT base,
       any_value(unit)       AS example_unit,
       count(DISTINCT indicator) AS indicators,
       any_value(weightable) AS weightable
FROM firm
GROUP BY base
ORDER BY indicators DESC
""", warn=False)

,base,example_unit,indicators,weightable
0,all_enterprises,PC_ENT,291,True
1,enterprises_with_internet_for_staff,PC_ENT_IUSE,230,False
2,enterprises_using_a_computer,PC_ENT_CUSE,88,False
3,enterprises_with_internet_access,PC_ENT_IACC,41,False
4,enterprises_buying_cloud_services,PC_ENT_CC,27,False
5,enterprises_using_ai,PC_ENT_AI_TANY,25,False
6,enterprises_with_web_sales,PC_ENT_AWSELL,20,False
7,enterprises_with_a_website,PC_ENT_WEB,14,False
8,enterprises_with_fixed_broadband,PC_ENT_FIXBB,14,False
9,enterprises_doing_own_data_analytics,PC_ENT_DAOWN,11,False


In [4]:
# Three numbers that look comparable and are not. Note the warning fires.
q("""
SELECT indicator, round(value,1) AS value, unit, base, weightable
FROM firm
WHERE geo='IT' AND time='2025' AND size_emp='SME_10_249'
  AND (indicator, unit) IN (('E_AI_TANY','PC_ENT'),
                            ('E_AI_PMS','PC_ENT_AI_TANY'),
                            ('E_AI_BLE','PC_ENT_AI_EC'))
""")

!! MIXED BASES (3): ['all_enterprises', 'enterprises_using_ai', 'enterprises_that_considered_ai']
   These values are percentages of different populations - not comparable.


,indicator,value,unit,base,weightable
0,E_AI_TANY,15.7,PC_ENT,all_enterprises,True
1,E_AI_PMS,33.1,PC_ENT_AI_TANY,enterprises_using_ai,False
2,E_AI_BLE,58.7,PC_ENT_AI_EC,enterprises_that_considered_ai,False


## Discovery — find indicators by meaning

~810 indicator/unit combinations. Search the labels rather than memorising codes; the result shows each one's base so you can tell at a glance what is comparable with what.

In [5]:
def find(term, table="firm", limit=15):
    """Search indicator labels for a phrase."""
    return q(f"""
        SELECT DISTINCT indicator, indic_label, unit, base
        FROM {table}
        WHERE lower(indic_label) LIKE '%{term.lower()}%'
        ORDER BY indicator, unit
        LIMIT {limit}
    """, warn=False)


find("artificial intelligence", limit=8)

,indicator,indic_label,unit,base
0,E_AI_BNU,"Enterprises do not use AI technologies, because artificial Intelligence technologies a...",PC_ENT,all_enterprises
1,E_AI_BNU,"Enterprises do not use AI technologies, because artificial Intelligence technologies a...",PC_ENT_AI_EC,enterprises_that_considered_ai
2,E_AI_BNU,"Enterprises do not use AI technologies, because artificial Intelligence technologies a...",PC_ENT_AI_TX,enterprises_not_using_ai
3,E_AI_BNU,"Enterprises do not use AI technologies, because artificial Intelligence technologies a...",PC_ENT_IUSE,enterprises_with_internet_for_staff
4,E_DI3_HI_AI_TANY,"Enterprises with high digital intensity index, which use any artificial intelligence t...",PC_ENT,all_enterprises
5,E_DI3_LO_AI_TANY,"Enterprises with low digital intensity index, which use any artificial intelligence te...",PC_ENT,all_enterprises
6,E_DI3_VHI_AI_TANY,"Enterprises with very high digital intensity index, which use any artificial intellige...",PC_ENT,all_enterprises
7,E_DI3_VLO_AI_TANY,"Enterprises with very low digital intensity index, which use any artificial intelligen...",PC_ENT,all_enterprises


## 1. Country ranking — where each country stands

`E_AI_TANY` = uses at least one AI technology, `PC_ENT` = share of all enterprises.

In [6]:
q("""
SELECT geo, geo_label, round(value, 1) AS sme_ai_pct
FROM firm
WHERE indicator = 'E_AI_TANY' AND unit = 'PC_ENT'
  AND time = '2025' AND size_emp = 'SME_10_249'
ORDER BY value DESC
""")

,geo,geo_label,sme_ai_pct
0,DK,Denmark,41.0
1,FI,Finland,36.4
2,SE,Sweden,33.6
3,BE,Belgium,33.0
4,LU,Luxembourg,32.8
5,NL,Netherlands,31.9
6,AT,Austria,28.7
7,NO,Norway,27.9
8,DE,Germany,24.9
9,EE,Estonia,22.7


## 2. The SME vs large gap

Ordered by gap size, not adoption — a country can have high adoption *and* a wide internal divide.

In [7]:
q("""
SELECT geo,
       round(MAX(CASE WHEN size_emp='SME_10_249'  THEN value END), 1) AS sme,
       round(MAX(CASE WHEN size_emp='LARGE_GE250' THEN value END), 1) AS large,
       round(MAX(CASE WHEN size_emp='LARGE_GE250' THEN value END)
           - MAX(CASE WHEN size_emp='SME_10_249'  THEN value END), 1) AS gap_pp,
       round(MAX(CASE WHEN size_emp='LARGE_GE250' THEN value END)
           / NULLIF(MAX(CASE WHEN size_emp='SME_10_249' THEN value END), 0), 1) AS times_ahead
FROM firm
WHERE indicator = 'E_AI_TANY' AND unit = 'PC_ENT' AND time = '2025'
GROUP BY geo
HAVING gap_pp IS NOT NULL
ORDER BY gap_pp DESC
""")

,geo,sme,large,gap_pp,times_ahead
0,SI,20.2,71.7,51.5,3.5
1,BE,33.0,76.4,43.4,2.3
2,FI,36.4,79.4,43.0,2.2
3,FR,17.0,58.0,41.0,3.4
4,AT,28.7,68.3,39.5,2.4
5,IE,18.3,57.3,38.9,3.1
6,PL,7.1,45.8,38.7,6.5
7,PT,10.7,49.2,38.5,4.6
8,ES,19.1,57.5,38.3,3.0
9,SE,33.6,71.9,38.2,2.1


## 3. Is the gap closing or widening?

Size classes overlap (`SME_10_249` contains `SMALL_10_49` + `MEDIUM_50_249`), so pick bands — never sum them.

In [8]:
q("""
SELECT time, geo,
       round(MAX(CASE WHEN size_emp='SME_10_249'  THEN value END), 1) AS sme,
       round(MAX(CASE WHEN size_emp='LARGE_GE250' THEN value END), 1) AS large,
       round(MAX(CASE WHEN size_emp='LARGE_GE250' THEN value END)
           - MAX(CASE WHEN size_emp='SME_10_249'  THEN value END), 1) AS gap_pp
FROM firm
WHERE indicator = 'E_AI_TANY' AND unit = 'PC_ENT'
  AND geo IN ('IT','EU27_2020','DK')
GROUP BY time, geo
HAVING gap_pp IS NOT NULL
ORDER BY geo, time
""")

,time,geo,sme,large,gap_pp
0,2021,DK,22.6,66.2,43.6
1,2023,DK,14.1,51.4,37.4
2,2024,DK,26.4,63.4,37.0
3,2025,DK,41.0,74.5,33.5
4,2021,EU27_2020,7.1,28.4,21.4
5,2023,EU27_2020,7.4,30.5,23.1
6,2024,EU27_2020,12.6,41.2,28.5
7,2025,EU27_2020,18.9,55.0,36.1
8,2021,IT,5.8,24.3,18.5
9,2023,IT,4.7,24.1,19.4


## 4. Absolute numbers

`AND weightable` is doing real work here: `enterprise_count` counts *all* enterprises, so only `PC_ENT` shares its base. Weighting a barrier or purpose indicator by it would produce a meaningless figure. Restricted to 2021–2024, where the universe exists.

These are **estimates** — the survey population and the SBS business economy are not identical scopes.

In [9]:
q("""
SELECT geo,
       round(value, 1) AS pct,
       enterprise_count::BIGINT AS smes,
       round(value / 100 * enterprise_count)::BIGINT         AS using_ai,
       round((100 - value) / 100 * enterprise_count)::BIGINT AS not_using_ai
FROM firm
WHERE indicator = 'E_AI_TANY' AND weightable
  AND time = '2024' AND size_emp = 'SME_10_249'
  AND enterprise_count IS NOT NULL
ORDER BY not_using_ai DESC
LIMIT 12
""")

,geo,pct,smes,using_ai,not_using_ai
0,EU27_2020,12.6,1820617,230126,1590491
1,DE,18.8,480442,90227,390215
2,IT,7.7,236333,18292,218041
3,FR,9.3,192646,17820,174826
4,ES,10.3,186085,19167,166918
5,PL,4.9,108625,5344,103281
6,RO,2.8,52667,1464,51203
7,PT,7.9,55086,4335,50751
8,NL,21.9,64485,14122,50363
9,EL,9.5,54353,5180,49173


## 5. Barriers — why SMEs don't adopt

Base is `enterprises_that_considered_ai`, **not** all enterprises. These figures are therefore not comparable with the adoption percentages above, and cannot be weighted.

In [10]:
q("""
SELECT replace(indic_label, 'Enterprises do not use AI technologies, because ', '') AS barrier,
       round(MAX(CASE WHEN geo='IT' THEN value END), 1)        AS italy,
       round(MAX(CASE WHEN geo='EU27_2020' THEN value END), 1) AS eu27
FROM firm
WHERE indicator LIKE 'E_AI_B%' AND unit = 'PC_ENT_AI_EC'
  AND time = '2025' AND size_emp = 'SME_10_249'
GROUP BY barrier
ORDER BY eu27 DESC
""")

,barrier,italy,eu27
0,of a lack of relevant expertise,58.7,70.5
1,of a lack of clarity about the legal consequences,47.4,53.7
2,of concerns regarding violation of data protection and privacy,43.1,52.5
3,of difficulties with availability or quality of the necessary data,45.3,43.5
4,"of incompatibility with existing equipment, software or systems",38.2,41.8
5,the costs seem too high,43.1,38.5
6,of ethical considerations,25.9,24.8
7,artificial Intelligence technologies are not useful for Enterprise,14.9,18.1


## 6. Sectors

Sector rows are all-sizes (`GE10`) and mostly unweighted — the ICT survey's NACE groupings don't exist in the SBS universe. Percentages are unaffected.

In [11]:
q("""
SELECT nace_r2, nace_label, round(value, 1) AS ai_pct
FROM firm
WHERE indicator = 'E_AI_TANY' AND unit = 'PC_ENT'
  AND time = '2025' AND geo = 'EU27_2020' AND nace_r2 IS NOT NULL
ORDER BY value DESC
LIMIT 20
""")

,nace_r2,nace_label,ai_pct
0,J62_J63,"Computer programming, consultancy, and information service activities",65.7
1,J,Information and communication,62.5
2,ICT,Information and Communication Technology - total,60.1
3,J58-J60,"Publishing, motion picture, video, television programme production; sound recording, p...",56.2
4,M72,Scientific research and development,52.1
5,M73-M75,"Advertising and market research; other professional, scientific and technical activiti...",48.0
6,C21,Manufacture of basic pharmaceutical products and pharmaceutical preparations,41.3
7,M,"Professional, scientific and technical activities",40.4
8,J61,Telecommunications,39.8
9,L_M,"Real estate activities; professional, scientific and technical activities",38.2


## 7. The people side

Two sources: generative-AI use (2025) and digital skills (2021 / 2023 / 2025 — so a trend is finally possible).

These rows are **overlapping subsets**: the same person appears under `IND_TOTAL`, an age band, a sex band and an education band. Filter, never sum. And they describe people, not businesses — compare with the firm table, never join to it.

In [12]:
q("""
SELECT dataset, indicator, indic_label, base, count(*) AS rows
FROM people
WHERE unit = 'PC_IND'
GROUP BY dataset, indicator, indic_label, base
ORDER BY dataset, indicator
LIMIT 20
""", warn=False)

,dataset,indicator,indic_label,base,rows
0,isoc_ai_iaiu,I_IUAI,Use of generative AI tools: in the last 3 months,all_individuals,3115
1,isoc_ai_iaiu,I_IUAIFE,Use of generative AI tools: for formal education,all_individuals,2872
2,isoc_ai_iaiu,I_IUAIPR,Use of generative AI tools: for private purposes,all_individuals,2872
3,isoc_ai_iaiu,I_IUAIWP,Use of generative AI tools: for professional (work) purposes,all_individuals,2872
4,isoc_sk_dskl_i21,I_DSK2_AB,Individuals with above basic overall digital skills (all five component indicators are...,all_individuals,11356
5,isoc_sk_dskl_i21,I_DSK2_B,Individuals with basic overall digital skills (all five component indicators are at ba...,all_individuals,11356
6,isoc_sk_dskl_i21,I_DSK2_BAB,Individuals with basic or above basic overall digital skills (all five component indic...,all_individuals,11356
7,isoc_sk_dskl_i21,I_DSK2_CC_AB,Individuals with above basic communication and collaboration skills,all_individuals,11356
8,isoc_sk_dskl_i21,I_DSK2_CC_B,Individuals with basic communication and collaboration skills,all_individuals,11356
9,isoc_sk_dskl_i21,I_DSK2_CC_BAB,Individuals with basic or above basic communication and collaboration skills,all_individuals,11356


In [13]:
# Digital skills over time - the trend the individual table previously could not support
q("""
SELECT indic_label, time,
       round(MAX(CASE WHEN geo='IT' THEN value END), 1)        AS italy,
       round(MAX(CASE WHEN geo='EU27_2020' THEN value END), 1) AS eu27
FROM people
WHERE indicator IN ('I_DSK2_AB','I_DSK2_BAB') AND unit = 'PC_IND'
  AND ind_type = 'IND_TOTAL'
GROUP BY indic_label, time
ORDER BY indic_label, time
""")

,indic_label,time,italy,eu27
0,Individuals with above basic overall digital skills (all five component indicators are...,2021,22.5,26.5
1,Individuals with above basic overall digital skills (all five component indicators are...,2023,22.2,27.3
2,Individuals with above basic overall digital skills (all five component indicators are...,2025,31.5,31.4
3,Individuals with basic or above basic overall digital skills (all five component indic...,2021,45.6,53.9
4,Individuals with basic or above basic overall digital skills (all five component indic...,2023,45.8,55.6
5,Individuals with basic or above basic overall digital skills (all five component indic...,2025,54.3,60.4


In [14]:
# Shadow AI: people using genAI *for work* vs firms deploying it.
# Two independent surveys, deliberately compared - never merged.
workers = q("""
    SELECT geo, round(value, 1) AS workers_using_genai_at_work
    FROM people
    WHERE indicator = 'I_IUAIWP' AND unit = 'PC_IND'
      AND ind_type = 'SAL_SELF_FAM' AND time = '2025'
""")
firms = q("""
    SELECT geo, round(value, 1) AS smes_deploying_genai
    FROM firm
    WHERE indicator = 'E_AI_TNLG' AND unit = 'PC_ENT'
      AND size_emp = 'SME_10_249' AND time = '2025'
""")
gap = workers.merge(firms, on="geo")
gap["ratio"] = (gap.workers_using_genai_at_work / gap.smes_deploying_genai).round(1)
gap.sort_values("ratio", ascending=False).head(15)

,geo,workers_using_genai_at_work,smes_deploying_genai,ratio
8,EL,28.1,3.5,8.0
13,CY,30.8,4.3,7.2
30,AL,18.3,2.7,6.8
18,MT,37.0,5.9,6.3
22,PT,27.6,4.8,5.8
17,HU,18.3,3.9,4.7
2,BG,14.1,3.1,4.5
14,LV,19.9,4.5,4.4
15,LT,27.1,6.2,4.4
24,SI,22.9,5.5,4.2


## Scratch

`q("...")` runs SQL against `firm` and `people`, warning if a result mixes bases. `find("phrase")` locates indicators.

Four rules, all enforced or surfaced by the view:
1. Always pin `unit` — it is part of the key
2. Only compare values sharing a `base`
3. Only weight where `weightable` is true, and only for 2021–2024
4. Never `SUM` across `size_emp` or `ind_type` — they overlap

In [15]:
find("security", limit=10)

,indicator,indic_label,unit,base
0,E_AI_PITS,Enterprises using AI technologies for ICT security,PC_ENT,all_enterprises
1,E_AI_PITS,Enterprises using AI technologies for ICT security,PC_ENT_AI_TANY,enterprises_using_ai
2,E_AI_PITS,Enterprises using AI technologies for ICT security,PC_ENT_IUSE,enterprises_with_internet_for_staff
3,E_CC_PSEC,Enterprises using security software applications (as a paid CC service),PC_ENT,all_enterprises
4,E_CC_PSEC,Enterprises using security software applications (as a paid CC service),PC_ENT_CC,enterprises_buying_cloud_services
5,E_CC_PSEC,Enterprises using security software applications (as a paid CC service),PC_ENT_IUSE,enterprises_with_internet_for_staff
6,E_ITSEC3,The ICT security related activities are carried out by own employees or external suppl...,PC_ENT,all_enterprises
7,E_ITSEC3,The ICT security related activities are carried out by own employees or external suppl...,PC_ENT_CUSE,enterprises_using_a_computer
8,E_ITSEC3,The ICT security related activities are carried out by own employees or external suppl...,PC_ENT_IUSE,enterprises_with_internet_for_staff
9,E_ITSEC3EXT,The ICT security related activities are carried out by external suppliers,PC_ENT,all_enterprises
